# 02 | Data Cleaning & Validation

Validated missing values, duplicate records, data types, financial values, unique identifiers, and referential integrity across project, budget, expense, and milestone datasets to ensure reliable data.



In [1]:
from pathlib import Path
import os
import sys
import subprocess

# 01 | GitHub repository
REPOSITORY_URL = (
    "https://github.com/MayerAnnamaria/python-business-analytics-project.git"
)

# 02 | Local project directory
PROJECT_DIR = Path("/content/python-business-analytics-project")

# 03 | Clone repository if it does not exist
if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)],
        check=True
    )

# 04 | Install project dependencies
%pip install "pandas>=2.0,<3.0" "numpy>=1.24,<3.0" "matplotlib>=3.7,<4.0" "openpyxl>=3.1,<4.0" "pytest>=7,<10"


# 05 | Set working directory
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 06 | Configure input and output folders
INPUT_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# OPTIONAL: Uncomment to use private Excel workbooks in your Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_DIR = Path('/content/drive/MyDrive/project-budget-performance-analytics/input')
# NOTE: You must provide all four files with compatible columns.


In [2]:
from src.business_analytics.io import read_sources
from src.business_analytics.cleaning import quality_report, clean_sources
raw=read_sources(INPUT_DIR)
before=quality_report(raw)
display(before.sort_values(['missing_count','dataset'],ascending=[False,True]).head(20))
print('Duplicate rows by source:',{name:int(df.duplicated().sum()) for name,df in raw.items()})


,dataset,column,rows,missing_count,missing_pct,exact_duplicate_rows
22,milestones,actual_date,96,21,21.88,0
7,projects,actual_end_date,25,9,36.00,1
12,actual_costs,transaction_id,289,0,0.00,1
13,actual_costs,project_id,289,0,0.00,1
14,actual_costs,cost_category,289,0,0.00,1
15,actual_costs,transaction_date,289,0,0.00,1
16,actual_costs,actual_cost,289,0,0.00,1
17,actual_costs,description,289,0,0.00,1
9,budget_plan,project_id,96,0,0.00,0
10,budget_plan,cost_category,96,0,0.00,0


Duplicate rows by source: {'projects': 1, 'budget_plan': 0, 'actual_costs': 1, 'milestones': 0}


In [3]:
# DATA CLEANING & QUALITY VALIDATION
#########################################

# Clean the source datasets
tables = clean_sources(raw)

# Generate the data quality report after cleaning
after = quality_report(tables)

# Display the top 20 data quality results
display(
    after.sort_values(
        ["missing_count", "dataset"],
        ascending=[False, True]
    ).head(20)
)

# Display cleaned row counts
print("Cleaned row counts:", {
    name: len(df) for name, df in tables.items()
})



,dataset,column,rows,missing_count,missing_pct,exact_duplicate_rows
22,milestones,actual_date,96,21,21.88,0
7,projects,actual_end_date,24,9,37.50,0
12,actual_costs,transaction_id,288,0,0.00,0
13,actual_costs,project_id,288,0,0.00,0
14,actual_costs,cost_category,288,0,0.00,0
15,actual_costs,transaction_date,288,0,0.00,0
16,actual_costs,actual_cost,288,0,0.00,0
17,actual_costs,description,288,0,0.00,0
9,budget_plan,project_id,96,0,0.00,0
10,budget_plan,cost_category,96,0,0.00,0


Cleaned row counts: {'projects': 24, 'budget_plan': 96, 'actual_costs': 288, 'milestones': 96}


**Decision log:

| Issue | Decision | Rationale |
|-------|----------|-----------|
| Missing completion dates | Preserve for open projects. | Avoid misclassifying project status. |
| Invalid financial values | Reject rather than impute. | Ensure accurate budget calculations. |
| Invalid project IDs | Validate against project master data. | Maintain referential integrity. |
| Duplicate records | Remove exact duplicates and validate unique keys. | Prevent double counting. |
